In [ ]:
!pip install titans-pytorch transformers datasets -q


# Titans — Στόχος 2: Reasoning / BABILong-style  (v3)
### Paper: *Titans: Learning to Memorize at Test Time* — Behrouz, Zhong, Mirrokni (Google Research, 2025)

## Task: QA2 — Two Supporting Facts (multi-hop / "Lost in the Middle")

| Διαφάνεια "Στόχος 2" | Υλοποίηση |
|---|---|
| BABILong, Single/Two Supporting Facts | QA1 (sanity) + **QA2** (κύριο) |
| Κείμενα-θόρυβος + παρεμβολή facts | WikiText-2 + synthetic bAbI-style facts |
| Loss αποκλειστικά στην τελευταία λέξη | `restricted_loss_and_acc(logits, loc_ids, target_idx)` |
| 20% test split — άγνωστα τμήματα | 80/20 split WikiText-2 tokens |
| Αντοχή MAC vs Baseline | Accuracy vs gap × context length |

## Τι διορθώθηκε σε κάθε έκδοση (σύνοψη)

| Έκδοση | Πρόβλημα που εντοπίστηκε | Λύση |
|---|---|---|
| v1 | MAC `model(seq)` → data leakage (copy last token) | `model(seq[:,:-1])` για ΚΑΙ τα 2 |
| v2 | `transfer_weights` → MAC κολλούσε σε `loss=ln(6)` plateau | Αφαίρεση transfer_weights |
| **v3** | MAC κολλούσε ΠΑΛΙ σε `loss≈1.79` — 50257-way classification δεν μπορεί να μάθει cold-start | **Restricted CE**: `logits[:,loc_ids]` → 6-way loss |

## Κρίσιμες σχεδιαστικές αποφάσεις (τεκμηριωμένες στον κώδικα)

1. **Restricted Cross-Entropy** — και τα 2 μοντέλα έχουν full VOCAB_SIZE head,
   αλλά το loss υπολογίζεται πάνω στα 6 location logits.  
   `restr = logits[:, -1, :][:, loc_ids_t]  →  CE(restr, target_idx_0_5)`

2. **No data leakage** — seq μήκους `ctx+1`, είσοδος `seq[:,:-1]` για ΑΜΦΟΤΕΡΑ.

3. **Curriculum για MAC** — 128→256→512→1024, εναλλάσσοντας depths.  
   Χωρίς curriculum, cold-start στα 1024 tokens ήταν πολύ δύσκολο.

4. **SEP token** (GPT-2 EOS=50256) πριν από κάθε fact/ερώτηση.

5. **MAC εκπαιδεύεται από scratch** — no weight transfer από baseline.

## Χρήση
`MODE = "all"` → εκπαίδευση + αξιολόγηση (~2.5-3.5ώρες σε T4)  
`MODE = "train"` / `"eval"` → μόνο το αντίστοιχο μέρος


In [ ]:
# =============================================================================
# Titans: Learning to Memorize at Test Time — ΣΤΟΧΟΣ 2: Reasoning
# Paper: Behrouz, Zhong, Mirrokni (Google Research, 2025)
#
# ΤΙ ΑΛΛΑΞΕ ΑΠΟ v2 (βάσει πειραματικής ανάλυσης των logs):
#
# ΠΡΟΒΛΗΜΑ — MAC κολλούσε στο loss=ln(6)=1.79 (τυχαίο επίπεδο):
#   Αιτία: με vocab=50257, το μοντέλο δεν μαθαίνει γρήγορα ότι η απάντηση
#   ανήκει σε 6 συγκεκριμένα token IDs. Ο Baseline το μαθαίνει μέσω
#   curriculum (εύκολα short-context examples), ενώ το MAC αρχίζει κατευθείαν
#   στα 1024 tokens.
#
# ΛΥΣΗ 1 — Restricted Cross-Entropy (κλειδί):
#   Αντί loss(logits[50257], absolute_id), χρησιμοποιούμε:
#   loss(logits[:, loc_ids] [6 values], loc_index [0-5])
#   Ίδιο για ΑΜΦΟΤΕΡΑ τα μοντέλα — δίκαιη σύγκριση.
#   Τα μοντέλα εξακολουθούν να έχουν full VOCAB_SIZE head αλλά το gradient
#   signal επικεντρώνεται στις 6 σχετικές θέσεις.
#
# ΛΥΣΗ 2 — Curriculum για το MAC (ίδιο με baseline):
#   128→256→512→1024 tokens, εναλλάσσοντας depths.
#
# ΛΥΣΗ 3 — SEP token (GPT-2 EOS=50256) πριν από κάθε fact/ερώτηση:
#   Δίνει ξεκάθαρο σήμα "εδώ αρχίζει κάτι σημαντικό".
#
# ΑΝΑΛΛΟΙΩΤΟ:
#   seq μήκους ctx+1, model(seq[:,:-1]), target=seq[:,-1], no leakage.
#   Ίδιο parameter budget (Baseline 30.9M ≈ MAC dim=224 30.4M).
# =============================================================================

import os, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset
from titans_pytorch import MemoryAsContextTransformer

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
VOCAB_SIZE     = 50257
BASE_DIM       = 256
MAC_DIM        = 224        # ψηφισθεί ώστε params(MAC) ≤ params(Baseline)
DEPTH          = 4
SEGMENT_LEN    = 128
TRAIN_SEQ_LEN  = 1024
MAX_POS_EMB    = 8192
CHECKPOINT_DIR = "checkpoints_goal2_v3"
SEP_TOKEN      = 50256      # GPT-2 EOS — separator πριν από facts/ερωτήσεις

EVAL_CTX_LENGTHS = [1024, 2048, 4096]
EVAL_DEPTHS_QA1  = [0.1, 0.3, 0.5, 0.7, 0.9]
EVAL_GAPS_QA2    = [0.1, 0.3, 0.5, 0.7, 0.9]
DEPTH1_QA2       = 0.05
EVAL_SCHEDULE    = {1024: (4, 5), 2048: (4, 5), 4096: (2, 10)}

LOCATIONS = ["bathroom", "bedroom", "garden", "hallway", "kitchen", "office"]
OBJECTS   = ["football", "milk", "apple", "newspaper"]
TRAIN_DEPTHS = [0.1, 0.3, 0.5, 0.7, 0.9]   # depths κατά training (εναλλάσσονται)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
torch.manual_seed(42); random.seed(42); np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    print("\n" + "!"*60)
    print("  ⚠️  GPU not found. Colab: Runtime → T4 GPU → Save")
    print("!"*60 + "\n")
else:
    print(f"  ✅ GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print(f"  dim={BASE_DIM}/{MAC_DIM}, depth={DEPTH}, seg={SEGMENT_LEN}, "
      f"train_ctx={TRAIN_SEQ_LEN}")
print(f"  ln(6)={math.log(6):.4f} (τυχαίο επίπεδο)\n")

# ─────────────────────────────────────────────────────────────────────────────
# ΜΟΝΤΕΛΑ
# ─────────────────────────────────────────────────────────────────────────────

class BaselineTransformer(nn.Module):
    """Plain causal Transformer. Δέχεται seq[:,:-1], full VOCAB_SIZE head."""
    def __init__(self):
        super().__init__()
        self.embedding     = nn.Embedding(VOCAB_SIZE, BASE_DIM)
        self.pos_embedding = nn.Embedding(MAX_POS_EMB, BASE_DIM)
        layer = nn.TransformerEncoderLayer(
            d_model=BASE_DIM, nhead=8, dim_feedforward=BASE_DIM*4,
            batch_first=True, dropout=0.0)
        self.transformer = nn.TransformerEncoder(layer, num_layers=DEPTH)
        self.to_logits = nn.Linear(BASE_DIM, VOCAB_SIZE, bias=False)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        h   = self.embedding(x) + self.pos_embedding(pos)
        h   = self.transformer(h)
        return self.to_logits(h)   # (B, T, VOCAB_SIZE)


def build_mac():
    """Δημιουργεί MAC Titan με dim=MAC_DIM (≤ params του Baseline)."""
    return MemoryAsContextTransformer(
        num_tokens             = VOCAB_SIZE,
        dim                    = MAC_DIM,
        depth                  = DEPTH,
        segment_len            = SEGMENT_LEN,
        num_persist_mem_tokens = 8,
        num_longterm_mem_tokens= 16,
    ).to(device)


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# ─────────────────────────────────────────────────────────────────────────────
# RESTRICTED CROSS-ENTROPY (ΚΛΕΙΔΙ)
# ─────────────────────────────────────────────────────────────────────────────

def restricted_loss_and_acc(logits, loc_ids_t, target_idx):
    """
    Υπολογίζει loss και accuracy αποκλειστικά πάνω στις 6 location κλάσεις.

    logits:      (B, T, VOCAB_SIZE) — full vocab logits
    loc_ids_t:   LongTensor[6]      — absolute GPT-2 token IDs των locations
    target_idx:  LongTensor[B]      — index 0-5 του σωστού location

    Γιατί αυτό λύνει το πρόβλημα:
    Πριν: CrossEntropy(logits[50257], absolute_id) → πρέπει να ξεχωρίσει
    τη σωστή απάντηση από 50257 tokens. Το MAC κολλούσε σε ln(6)=1.79
    επειδή δεν μπορούσε να μάθει ποια 6 από τα 50257 tokens είναι σχετικά.
    Τώρα: CrossEntropy(logits[6], idx_0_5) → μαθαίνει να διακρίνει 6 κλάσεις.
    Το τυχαίο επίπεδο παραμένει ln(6)=1.79, αλλά το ceiling είναι εφικτό.
    Ισχύει ΚΑΙ για τα 2 μοντέλα → δίκαιη σύγκριση.
    """
    restr = logits[:, -1, :][:, loc_ids_t]       # (B, 6)
    loss  = nn.CrossEntropyLoss()(restr, target_idx)
    acc   = (restr.argmax(-1) == target_idx).float().mean()
    return loss, acc


# ─────────────────────────────────────────────────────────────────────────────
# ΔΕΔΟΜΕΝΑ
# ─────────────────────────────────────────────────────────────────────────────

def load_wiki_tokens(tokenizer):
    print("  Downloading WikiText-2...")
    ds   = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    txt  = "\n".join(ds["text"])
    toks = tokenizer.encode(txt, return_tensors="pt").squeeze(0)
    sp   = int(len(toks) * 0.8)
    tr, te = toks[:sp], toks[sp:]
    print(f"  Tokens: {len(toks):,} | Train: {len(tr):,} (80%) | Test: {len(te):,} (20%)")
    return tr, te


def _ins(seq, ids, pos):
    seq[pos : pos + len(ids)] = torch.tensor(ids, dtype=torch.long, device=device)


def make_qa1_batch(tok_pool, tokenizer, loc_ids, bs, ctx_len, depth_pct=None):
    """
    QA1 — Single Supporting Fact.
    seq: [noise... SEP fact noise... SEP Q ... ANS]
    Επιστρέφει seq (B,ctx+1) και target_idx (B,) ∈ {0..5}.
    """
    assert ctx_len % SEGMENT_LEN == 0
    seqs, targets = torch.zeros(bs, ctx_len+1, dtype=torch.long, device=device), []
    q_ids = [SEP_TOKEN] + tokenizer.encode(" Question where is Mary answer")
    qlen  = len(q_ids) + 1   # +1 για το answer token

    for b in range(bs):
        start = random.randint(0, max(0, len(tok_pool) - ctx_len - 2))
        seq   = tok_pool[start : start + ctx_len + 1].clone().to(device)
        li    = random.randrange(len(LOCATIONS))
        f_ids = [SEP_TOKEN] + tokenizer.encode(f" Mary went to the {LOCATIONS[li]}.")
        d     = depth_pct if depth_pct is not None else random.uniform(0.05, 0.90)
        max_i = ctx_len - len(f_ids) - qlen - 2
        pos   = max(0, min(int(ctx_len * d), max_i))
        _ins(seq, f_ids, pos)
        seq[-(qlen):-1] = torch.tensor(q_ids, dtype=torch.long, device=device)
        seq[-1]         = loc_ids[li]
        seqs[b] = seq; targets.append(li)

    return seqs, torch.tensor(targets, dtype=torch.long, device=device)


def make_qa2_batch(tok_pool, tokenizer, loc_ids, bs, ctx_len,
                   depth1=None, depth2=None):
    """
    QA2 — Two Supporting Facts (multi-hop).
    Fact1 (depth1): "Mary picked up the {obj}."
    Fact2 (depth2): "Mary went to the {loc}."
    Q: "Where is the {obj}?" → Target idx of loc.
    """
    assert ctx_len % SEGMENT_LEN == 0
    seqs, targets = torch.zeros(bs, ctx_len+1, dtype=torch.long, device=device), []

    for b in range(bs):
        start = random.randint(0, max(0, len(tok_pool) - ctx_len - 2))
        seq   = tok_pool[start : start + ctx_len + 1].clone().to(device)
        oi = random.randrange(len(OBJECTS)); li = random.randrange(len(LOCATIONS))
        obj, loc = OBJECTS[oi], LOCATIONS[li]
        f1 = [SEP_TOKEN] + tokenizer.encode(f" Mary picked up the {obj}.")
        f2 = [SEP_TOKEN] + tokenizer.encode(f" Mary went to the {loc}.")
        q  = [SEP_TOKEN] + tokenizer.encode(f" Question where is the {obj} answer")
        ql = len(q) + 1

        d1 = DEPTH1_QA2 if depth1 is None else depth1
        d2 = (random.uniform(d1 + 0.1, 0.92) if depth2 is None
              else min(depth2, 0.95))
        max_i = ctx_len - max(len(f1), len(f2)) - ql - 4
        p1 = max(0, min(int(ctx_len * d1), max_i - len(f1) - 2))
        p2 = max(p1 + len(f1) + 2, min(int(ctx_len * d2), max_i))
        _ins(seq, f1, p1); _ins(seq, f2, p2)
        seq[-ql:-1] = torch.tensor(q, dtype=torch.long, device=device)
        seq[-1]     = loc_ids[li]
        seqs[b] = seq; targets.append(li)

    return seqs, torch.tensor(targets, dtype=torch.long, device=device)


# ─────────────────────────────────────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def _run_loop(model, batch_fn, label, loc_ids_t, total_iters, lr,
              ckpt_name, accum=4, min_iters=0, stop_loss=0.05, stop_acc=0.95):
    opt    = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    n_upd  = max(1, total_iters // accum)
    warmup = max(5, n_upd // 8)
    sch    = get_linear_schedule_with_warmup(opt, warmup, n_upd)
    model.train(); opt.zero_grad()
    best_loss, best_state = float("inf"), None
    log_every = max(1, total_iters // 20)

    print(f"\n{'='*60}")
    print(f"  [{label}] iters={total_iters} lr={lr} accum={accum} "
          f"warmup={warmup} upd={n_upd}")
    print(f"{'='*60}")

    for i in range(total_iters):
        seq, tgt  = batch_fn()
        logits    = model(seq[:, :-1])
        loss, acc = restricted_loss_and_acc(logits, loc_ids_t, tgt)
        (loss / accum).backward()
        if (i+1) % accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
        if device.type == "cuda": torch.cuda.empty_cache()

        if i % log_every == 0 or i == total_iters - 1:
            rl = loss.item()
            print(f"  Iter {i:04d}/{total_iters} | loss={rl:.4f} | "
                  f"acc={acc.item()*100:.1f}% | lr={sch.get_last_lr()[0]:.2e}")
            if rl < best_loss:
                best_loss = rl
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if rl < stop_loss and acc.item() > stop_acc and i >= min_iters:
                print(f"  ✅ Converged at iter {i}."); break

    if best_state: model.load_state_dict(best_state)
    p = os.path.join(CHECKPOINT_DIR, ckpt_name)
    torch.save(model.state_dict(), p)
    print(f"  Saved (best loss={best_loss:.4f}) → {p}")


def _curriculum(model, train_tok, tokenizer, loc_ids, loc_ids_t,
                stages, label, ckpt_name):
    """Curriculum training: εναλλάσσει TRAIN_DEPTHS σε κάθε batch."""
    opt     = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    n_total = sum(s[2] for s in stages)
    n_upd   = max(1, n_total // 4)
    sch     = get_linear_schedule_with_warmup(opt, max(5, n_upd//8), n_upd)
    accum   = 4; opt.zero_grad(); model.train()
    best_loss, best_state, giter = float("inf"), None, 0

    for ctx, bs, max_it in stages:
        print(f"\n  Stage ctx={ctx} bs={bs} iters={max_it}")
        for local_i in range(max_it):
            d = TRAIN_DEPTHS[local_i % len(TRAIN_DEPTHS)]
            seq, tgt  = make_qa1_batch(train_tok, tokenizer, loc_ids, bs, ctx, d)
            logits    = model(seq[:, :-1])
            loss, acc = restricted_loss_and_acc(logits, loc_ids_t, tgt)
            (loss / accum).backward()
            if (giter+1) % accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad()
            if giter % 200 == 0:
                rl = loss.item()
                print(f"  Iter {giter:04d} | loss={rl:.4f} | "
                      f"acc={acc.item()*100:.1f}% | ctx={ctx}")
                if rl < best_loss:
                    best_loss = rl
                    best_state = {k: v.clone() for k,v in model.state_dict().items()}
                if rl < 0.05 and acc.item() > 0.98 and ctx == TRAIN_SEQ_LEN:
                    print("  ✅ QA1 converged early."); break
            giter += 1
        if device.type == "cuda": torch.cuda.empty_cache()

    if best_state: model.load_state_dict(best_state)
    p = os.path.join(CHECKPOINT_DIR, ckpt_name)
    torch.save(model.state_dict(), p); print(f"  Saved → {p}")


# ─────────────────────────────────────────────────────────────────────────────
# ΕΚΠΑΙΔΕΥΣΗ
# ─────────────────────────────────────────────────────────────────────────────

def train_baseline(model, train_tok, tokenizer, loc_ids):
    loc_ids_t = torch.tensor(loc_ids, dtype=torch.long, device=device)
    print("\n" + "#"*60)
    print("  BASELINE — QA1 Curriculum + QA2 Fine-tune")
    print("#"*60)
    _curriculum(model, train_tok, tokenizer, loc_ids, loc_ids_t,
                stages=[(128,16,500),(256,12,500),(512,8,500),(1024,6,2000)],
                label="Baseline", ckpt_name="baseline_qa1.pt")
    print("\n  ── QA2 Fine-tuning ──")
    _run_loop(model,
              lambda: make_qa2_batch(train_tok, tokenizer, loc_ids, 6, TRAIN_SEQ_LEN),
              "Baseline-QA2", loc_ids_t, 1500, 1e-4,
              "baseline_qa2.pt", accum=4, min_iters=500)


def train_mac(mac_model, train_tok, tokenizer, loc_ids):
    loc_ids_t = torch.tensor(loc_ids, dtype=torch.long, device=device)
    print("\n" + "#"*60)
    print("  MAC TITAN — QA1 Curriculum + QA2 Fine-tune")
    print("  (from scratch, restricted cross-entropy, curriculum 128→1024)")
    print("#"*60)
    _curriculum(mac_model, train_tok, tokenizer, loc_ids, loc_ids_t,
                stages=[(128,8,600),(256,6,600),(512,4,600),(1024,4,3000)],
                label="MAC", ckpt_name="mac_qa1.pt")
    print("\n  ── QA2 Fine-tuning ──")
    _run_loop(mac_model,
              lambda: make_qa2_batch(train_tok, tokenizer, loc_ids, 4, TRAIN_SEQ_LEN),
              "MAC-QA2", loc_ids_t, 2500, 1e-4,
              "mac_qa2.pt", accum=4, min_iters=1000)


# ─────────────────────────────────────────────────────────────────────────────
# ΑΞΙΟΛΟΓΗΣΗ
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def _eval(model, label, task, test_tok, tokenizer, loc_ids,
          ctx_lengths, y_vals):
    loc_ids_t = torch.tensor(loc_ids, dtype=torch.long, device=device)
    print(f"\n{'='*60}\n  [{task}] {label}\n{'='*60}")
    model.eval()
    R = np.zeros((len(y_vals), len(ctx_lengths)))
    for ci, ctx in enumerate(ctx_lengths):
        bs, n_tr = EVAL_SCHEDULE[ctx]
        for yi, y in enumerate(y_vals):
            correct = total = 0
            for _ in range(n_tr):
                if task == "QA1":
                    seq, tgt = make_qa1_batch(test_tok, tokenizer, loc_ids, bs, ctx, y)
                else:
                    d2 = min(DEPTH1_QA2 + y, 0.97)
                    seq, tgt = make_qa2_batch(test_tok, tokenizer, loc_ids,
                                               bs, ctx, DEPTH1_QA2, d2)
                restr  = model(seq[:, :-1])[:, -1, :][:, loc_ids_t]
                preds  = restr.argmax(-1)
                correct += (preds == tgt).sum().item()
                total   += bs
                if device.type == "cuda": torch.cuda.empty_cache()
            acc = correct / total; R[yi, ci] = acc
            mark = "✅" if acc>=0.7 else ("~" if acc>=0.4 else "❌")
            extra = f"depth={y:.1f}" if task=="QA1" else f"gap={y:.1f} (d2={min(DEPTH1_QA2+y,0.97):.2f})"
            print(f"  {mark} ctx={ctx:5d} | {extra} → {acc*100:.1f}%")
    return R


def evaluate(model, label, test_tok, tokenizer, loc_ids):
    qa1 = _eval(model, label, "QA1", test_tok, tokenizer, loc_ids,
                EVAL_CTX_LENGTHS, EVAL_DEPTHS_QA1)
    qa2 = _eval(model, label, "QA2", test_tok, tokenizer, loc_ids,
                EVAL_CTX_LENGTHS, EVAL_GAPS_QA2)
    return qa1, qa2


# ─────────────────────────────────────────────────────────────────────────────
# ΑΠΟΤΕΛΕΣΜΑΤΑ & PLOTS
# ─────────────────────────────────────────────────────────────────────────────

def print_table(bq1, mq1, bq2, mq2):
    xl = "  ".join(f"{c//1024}K" for c in EVAL_CTX_LENGTHS)
    for name, bm, mm, ys, yl in [
        ("QA1 — Single Fact",          bq1, mq1, EVAL_DEPTHS_QA1, "depth"),
        ("QA2 — Two Facts (Lost-in-Middle)", bq2, mq2, EVAL_GAPS_QA2, "gap"),
    ]:
        print(f"\n{'='*66}\n  {name}\n{'='*66}")
        print(f"  {yl:>5} | {'Baseline':^26} | {'MAC Titan':^26}")
        print(f"  {'':5} | {xl:^26} | {xl:^26}")
        print(f"  {'-'*66}")
        for yi, y in enumerate(ys):
            br = "  ".join(f"{bm[yi,ci]*100:5.1f}%" for ci in range(len(EVAL_CTX_LENGTHS)))
            mr = "  ".join(f"{mm[yi,ci]*100:5.1f}%" for ci in range(len(EVAL_CTX_LENGTHS)))
            print(f"  {y:.1f}   | {br} | {mr}")


def _hm(ax, data, title, xl, yl, xlabel, ylabel):
    im = ax.imshow(data, cmap="RdYlGn", vmin=0, vmax=1,
                   origin="lower", aspect="auto", interpolation="nearest")
    ax.set(xticks=range(len(xl)), yticks=range(len(yl)))
    ax.set_xticklabels(xl); ax.set_yticklabels([f"{v:.1f}" for v in yl])
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title, fontsize=10)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i,j]
            ax.text(j, i, f"{v*100:.0f}%", ha="center", va="center",
                    fontsize=9, color="black" if 0.25<v<0.75 else "white",
                    fontweight="bold")
    return im


def save_plots(bq1, mq1, bq2, mq2):
    xl = [f"{c//1024}K" for c in EVAL_CTX_LENGTHS]

    for data_pair, ylabels, ylabel, fname, title in [
        ((bq1, mq1), EVAL_DEPTHS_QA1, "Fact depth (%)",
         "qa1_heatmap.png",
         "QA1 — Single Supporting Fact (Sanity Check)\nAccuracy ανά depth × context"),
        ((bq2, mq2), EVAL_GAPS_QA2, "Gap between facts (%)",
         "qa2_heatmap.png",
         "QA2 — Two Supporting Facts ('Lost in the Middle')\nAccuracy ανά gap × context"),
    ]:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(title, fontweight="bold")
        for ax, mat, ttl in zip(axes, data_pair, ["Baseline Transformer", "MAC Titan"]):
            im = _hm(ax, mat, ttl, xl, ylabels, "Context length", ylabel)
        fig.colorbar(im, ax=axes.tolist(), label="Accuracy")
        fig.savefig(fname, dpi=150, bbox_inches="tight"); plt.close(fig)
        print(f"  Saved → {fname}")

    # Lost-in-the-Middle curves
    n = len(EVAL_CTX_LENGTHS)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4), sharey=True)
    if n == 1: axes = [axes]
    fig.suptitle("QA2 — 'Lost in the Middle': Accuracy vs gap\n"
                 "(fact1 @ 5%, fact2 @ 5%+gap)", fontweight="bold")
    for j, ctx in enumerate(EVAL_CTX_LENGTHS):
        ax = axes[j]
        ax.plot(EVAL_GAPS_QA2, bq2[:, j], "o--", color="#e74c3c",
                lw=2, ms=7, label="Baseline Transformer")
        ax.plot(EVAL_GAPS_QA2, mq2[:, j], "o-",  color="#2ecc71",
                lw=2, ms=7, label="MAC Titan")
        ax.axhline(1/6, color="#888", lw=1.2, ls=":", label="Τυχαίο (1/6)")
        ax.set_title(f"context={ctx//1024}K tokens")
        ax.set_xlabel("Gap (depth2-depth1)"); ax.set_ylim(-0.05, 1.05)
        ax.grid(alpha=0.3); ax.set_xticks(EVAL_GAPS_QA2)
        if j == 0: ax.set_ylabel("Accuracy"); ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("qa2_lost_in_middle.png", dpi=150, bbox_inches="tight"); plt.close(fig)
    print("  Saved → qa2_lost_in_middle.png")

    # Summary bars
    bm = bq2.mean(0)*100; mm = mq2.mean(0)*100
    x  = np.arange(n); w = 0.32
    fig, ax = plt.subplots(figsize=(7, 4))
    b1 = ax.bar(x-w/2, bm, w, label="Baseline", color="#e74c3c", alpha=0.9)
    b2 = ax.bar(x+w/2, mm, w, label="MAC Titan", color="#2ecc71", alpha=0.9)
    ax.axhline(100/6, color="#888", lw=1.5, ls=":", label="Τυχαίο (17%)")
    for bar in list(b1)+list(b2):
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+1, f"{h:.0f}%",
                ha="center", fontsize=9, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels([f"{c//1024}K" for c in EVAL_CTX_LENGTHS])
    ax.set_xlabel("Context length (tokens)"); ax.set_ylabel("Mean Accuracy (%)")
    ax.set_ylim(0, 115)
    ax.set_title("QA2 — Mean accuracy ανά context length (avg over all gaps)")
    ax.legend(); fig.tight_layout()
    fig.savefig("qa2_summary.png", dpi=150, bbox_inches="tight"); plt.close(fig)
    print("  Saved → qa2_summary.png")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main(MODE="all"):
    print(f"\n{'='*60}")
    print(f"  Titans Goal-2 v3 | MODE={MODE} | device={device}")
    print(f"{'='*60}\n")

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    loc_ids   = [tokenizer.encode(f" {loc}")[0] for loc in LOCATIONS]
    print("  Location IDs:", dict(zip(LOCATIONS, loc_ids)))

    baseline = BaselineTransformer().to(device)
    mac      = build_mac()

    print(f"\n  Baseline params : {count_params(baseline):,}")
    print(f"  MAC Titan params: {count_params(mac):,}  (dim={MAC_DIM})\n")

    if MODE in ("train", "all"):
        tr, _ = load_wiki_tokens(tokenizer)
        train_baseline(baseline, tr, tokenizer, loc_ids)
        train_mac(mac, tr, tokenizer, loc_ids)
    else:
        for name, model, fname in [
            ("Baseline", baseline, "baseline_qa2.pt"),
            ("MAC",      mac,      "mac_qa2.pt"),
        ]:
            p = os.path.join(CHECKPOINT_DIR, fname)
            if not os.path.exists(p):
                raise FileNotFoundError(f"{p} δεν βρέθηκε. Τρέξε πρώτα train.")
            model.load_state_dict(torch.load(p, map_location=device))
            print(f"  ✅ {name} ← {p}")

    if MODE in ("eval", "all"):
        _, te = load_wiki_tokens(tokenizer)

        bq1, bq2 = evaluate(baseline, "Baseline Transformer", te, tokenizer, loc_ids)
        mq1, mq2 = evaluate(mac,      "MAC Titan",            te, tokenizer, loc_ids)

        for nm, arr in [("base_qa1",bq1),("mac_qa1",mq1),
                        ("base_qa2",bq2),("mac_qa2",mq2)]:
            np.save(os.path.join(CHECKPOINT_DIR, f"{nm}.npy"), arr)

        print_table(bq1, mq1, bq2, mq2)
        save_plots(bq1, mq1, bq2, mq2)

        print(f"\n  ✅ Ολοκλήρωση.")
        print(f"  Plots: qa1_heatmap.png, qa2_heatmap.png, "
              f"qa2_lost_in_middle.png, qa2_summary.png")
        print(f"  .npy + .pt: {CHECKPOINT_DIR}/")


# Εκτέλεση: βλ. επόμενο cell


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ΕΚΤΕΛΕΣΗ
#   "all"   → εκπαίδευση + αξιολόγηση  (~2.5-3.5 ώρες σε T4 GPU)
#   "train" → μόνο εκπαίδευση (αποθηκεύει checkpoints στο checkpoints_goal2_v3/)
#   "eval"  → μόνο αξιολόγηση (χρειάζεται checkpoints από train)
# ─────────────────────────────────────────────────────────────────────────────
MODE = "all"
main(MODE)

## (Προαιρετικό) Αποθήκευση στο Google Drive

In [ ]:
from google.colab import drive
import shutil, datetime

drive.mount("/content/drive")
ts   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
dest = f"/content/drive/MyDrive/Titans_Goal2_v3_{ts}"
os.makedirs(dest, exist_ok=True)

for f in ["qa1_heatmap.png","qa2_heatmap.png",
          "qa2_lost_in_middle.png","qa2_summary.png"]:
    if os.path.exists(f):
        shutil.copy2(f, dest)

if os.path.isdir(CHECKPOINT_DIR):
    shutil.copytree(CHECKPOINT_DIR,
                    os.path.join(dest, CHECKPOINT_DIR),
                    dirs_exist_ok=True)
print(f"Αποθηκεύτηκαν στο: {dest}")